# Statistical Similarity Test between Real and Synthetic Data

This notebook performs statistical similarity testing between real and synthetic data.

## Objectives:
1. Load three datasets (Bank, Cancer, Alzhimers) from xlsx/csv files
2. Generate synthetic data using SDV with four models:
   - CTGAN
   - CopulaGAN
   - Gaussian Copula
   - TVAE
3. Evaluate statistical measures:
   - Mean
   - Median
   - Standard Deviation
   - Outliers (using IQR method)

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    GaussianCopulaSynthesizer,
    TVAESynthesizer
)

warnings.filterwarnings('ignore')


## Helper Functions


In [2]:
def detect_outliers_iqr(data, column):
    """
    Detect outliers using the Interquartile Range (IQR) method.
    
    Parameters:
    -----------
    data : pd.Series
        Data column to analyze
    column : str
        Column name
        
    Returns:
    --------
    int : Number of outliers
    """
    if not pd.api.types.is_numeric_dtype(data):
        return 0
    
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    return len(outliers)

def calculate_statistics(df, dataset_name, data_type='real'):
    """
    Calculate statistical measures for a dataset.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataset to analyze
    dataset_name : str
        Name of the dataset
    data_type : str
        'real' or 'synthetic'
        
    Returns:
    --------
    dict : Dictionary containing statistics
    """
    stats = {
        'dataset': dataset_name,
        'data_type': data_type,
        'statistics': {}
    }
    
    # Get numeric columns only
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if not numeric_cols:
        print(f"Warning: No numeric columns found in {dataset_name} ({data_type})")
        return stats
    
    for col in numeric_cols:
        col_data = df[col].dropna()
        
        if len(col_data) == 0:
            continue
        
        stats['statistics'][col] = {
            'mean': float(col_data.mean()),
            'median': float(col_data.median()),
            'std': float(col_data.std()),
            'outliers': detect_outliers_iqr(col_data, col),
            'count': len(col_data)
        }
    
    return stats

def compare_statistics(real_stats, synth_stats):
    """
    Compare statistics between real and synthetic data.
    
    Parameters:
    -----------
    real_stats : dict
        Statistics from real data
    synth_stats : dict
        Statistics from synthetic data
        
    Returns:
    --------
    dict : Comparison results
    """
    comparison = {
        'mean_diff': {},
        'median_diff': {},
        'std_diff': {},
        'outlier_diff': {},
        'mean_relative_error': {},
        'median_relative_error': {},
        'std_relative_error': {}
    }
    
    real_cols = set(real_stats['statistics'].keys())
    synth_cols = set(synth_stats['statistics'].keys())
    common_cols = real_cols.intersection(synth_cols)
    
    for col in common_cols:
        real = real_stats['statistics'][col]
        synth = synth_stats['statistics'][col]
        
        # Absolute differences
        comparison['mean_diff'][col] = abs(real['mean'] - synth['mean'])
        comparison['median_diff'][col] = abs(real['median'] - synth['median'])
        comparison['std_diff'][col] = abs(real['std'] - synth['std'])
        comparison['outlier_diff'][col] = abs(real['outliers'] - synth['outliers'])
        
        # Relative errors (percentage)
        if real['mean'] != 0:
            comparison['mean_relative_error'][col] = (
                abs(real['mean'] - synth['mean']) / abs(real['mean']) * 100
            )
        else:
            comparison['mean_relative_error'][col] = 0
        
        if real['median'] != 0:
            comparison['median_relative_error'][col] = (
                abs(real['median'] - synth['median']) / abs(real['median']) * 100
            )
        else:
            comparison['median_relative_error'][col] = 0
        
        if real['std'] != 0:
            comparison['std_relative_error'][col] = (
                abs(real['std'] - synth['std']) / abs(real['std']) * 100
            )
        else:
            comparison['std_relative_error'][col] = 0
    
    return comparison


## Load Datasets


In [3]:
datasets_path = Path('Datasets')
datasets = {}

# Load Bank dataset (CSV)
bank_path = datasets_path / 'bank-full.csv'
if bank_path.exists():
    print(f"Loading Bank dataset from {bank_path}...")
    bank_data = pd.read_csv(bank_path, sep=';')
    # Sample first 10,000 rows for faster processing
    if len(bank_data) > 10000:
        bank_data = bank_data.head(10000).reset_index(drop=True)
    datasets['Bank'] = bank_data
    print(f"Bank dataset loaded: {bank_data.shape}")
else:
    print(f"Warning: Bank dataset not found at {bank_path}")

# Load Cancer dataset (CSV)
cancer_path = datasets_path / 'Cancer.csv'
if cancer_path.exists():
    print(f"Loading Cancer dataset from {cancer_path}...")
    cancer_data = pd.read_csv(cancer_path)
    # Drop non-feature columns if they exist
    drop_cols = [c for c in ['id', 'ID', 'Unnamed: 0', 'Unnamed: 32'] if c in cancer_data.columns]
    if drop_cols:
        cancer_data = cancer_data.drop(columns=drop_cols)
    # Convert diagnosis to numeric if needed
    if 'diagnosis' in cancer_data.columns:
        cancer_data.replace({'diagnosis': {'M': 1, 'B': 0}}, inplace=True)
    datasets['Cancer'] = cancer_data
    print(f"Cancer dataset loaded: {cancer_data.shape}")
else:
    print(f"Warning: Cancer dataset not found at {cancer_path}")

# Load Alzhimers dataset (XLSX)
alzhimers_path = datasets_path / 'Alzhimers.xlsx'
if alzhimers_path.exists():
    print(f"Loading Alzhimers dataset from {alzhimers_path}...")
    alzhimers_data = pd.read_excel(alzhimers_path)
    # Drop non-feature columns if they exist
    drop_cols = [c for c in ['Subject ID', 'MRI ID', 'Hand', 'M/F', 'Group'] if c in alzhimers_data.columns]
    if drop_cols:
        alzhimers_data = alzhimers_data.drop(columns=drop_cols)
    # Handle missing values
    alzhimers_data = alzhimers_data.fillna(alzhimers_data.median(numeric_only=True))
    datasets['Alzhimers'] = alzhimers_data
    print(f"Alzhimers dataset loaded: {alzhimers_data.shape}")
else:
    print(f"Warning: Alzhimers dataset not found at {alzhimers_path}")

print(f"\nTotal datasets loaded: {len(datasets)}")


Loading Bank dataset from Datasets\bank-full.csv...
Bank dataset loaded: (10000, 17)
Loading Cancer dataset from Datasets\Cancer.csv...
Cancer dataset loaded: (569, 31)
Loading Alzhimers dataset from Datasets\Alzhimers.xlsx...
Alzhimers dataset loaded: (373, 10)

Total datasets loaded: 3


## Generate Synthetic Data and Evaluate

This section will:
1. Generate synthetic data for each dataset using all four SDV models
2. Calculate statistics for real and synthetic data
3. Compare the statistics


In [4]:
# Define models
models = {
    'CTGAN': CTGANSynthesizer,
    'CopulaGAN': CopulaGANSynthesizer,
    'GaussianCopula': GaussianCopulaSynthesizer,
    'TVAE': TVAESynthesizer
}

# Store all results
all_results = {}


In [5]:
# Process each dataset
for dataset_name, real_data in datasets.items():
    print(f"\n{'#'*60}")
    print(f"Processing Dataset: {dataset_name}")
    print(f"{'#'*60}")
    
    # Calculate statistics for real data
    print(f"\nCalculating statistics for real {dataset_name} data...")
    real_stats = calculate_statistics(real_data, dataset_name, 'real')
    
    # Create metadata
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(real_data)
    
    # Generate synthetic data for each model
    synthetic_data = {}
    num_samples = len(real_data)
    
    print(f"\n{'='*60}")
    print(f"Generating synthetic data for {dataset_name}")
    print(f"{'='*60}")
    
    for model_name, ModelClass in models.items():
        print(f"\nFitting {model_name} on {dataset_name}...")
        try:
            model = ModelClass(metadata=metadata)
            model.fit(real_data)
            
            print(f"Sampling {num_samples} rows from {model_name}...")
            synthetic_data[model_name] = model.sample(num_rows=num_samples)
            
            print(f"{model_name} completed successfully!")
        except Exception as e:
            print(f"Error with {model_name}: {str(e)}")
            synthetic_data[model_name] = None
    
    # Store results for this dataset
    dataset_results = {
        'real_stats': real_stats,
        'synthetic_data': {},
        'synthetic_stats': {},
        'comparisons': {}
    }
    
    # Evaluate each synthetic dataset
    for model_name, synth_data in synthetic_data.items():
        if synth_data is None:
            continue
        
        print(f"\nEvaluating {model_name} for {dataset_name}...")
        
        # Calculate statistics for synthetic data
        synth_stats = calculate_statistics(
            synth_data, dataset_name, f'synthetic_{model_name}'
        )
        
        # Compare statistics
        comparison = compare_statistics(real_stats, synth_stats)
        
        dataset_results['synthetic_data'][model_name] = synth_data
        dataset_results['synthetic_stats'][model_name] = synth_stats
        dataset_results['comparisons'][model_name] = comparison
    
    all_results[dataset_name] = dataset_results
    print(f"\n{'#'*60}")
    print(f"Completed processing {dataset_name}")
    print(f"{'#'*60}")



############################################################
Processing Dataset: Bank
############################################################

Calculating statistics for real Bank data...

Generating synthetic data for Bank

Fitting CTGAN on Bank...
Sampling 10000 rows from CTGAN...
CTGAN completed successfully!

Fitting CopulaGAN on Bank...
Sampling 10000 rows from CopulaGAN...
CopulaGAN completed successfully!

Fitting GaussianCopula on Bank...
Sampling 10000 rows from GaussianCopula...
GaussianCopula completed successfully!

Fitting TVAE on Bank...
Sampling 10000 rows from TVAE...
TVAE completed successfully!

Evaluating CTGAN for Bank...

Evaluating CopulaGAN for Bank...

Evaluating GaussianCopula for Bank...

Evaluating TVAE for Bank...

############################################################
Completed processing Bank
############################################################

############################################################
Processing Dataset: Cancer
####

## Results Summary


In [6]:
print("="*80)
print("STATISTICAL SIMILARITY TEST SUMMARY")
print("="*80)

for dataset_name, results in all_results.items():
    print(f"\n{'='*80}")
    print(f"Dataset: {dataset_name}")
    print(f"{'='*80}")
    
    real_stats = results['real_stats']
    print(f"\nReal Data Statistics:")
    print(f"  Number of numeric columns: {len(real_stats['statistics'])}")
    
    for model_name, comparison in results['comparisons'].items():
        print(f"\n{'-'*60}")
        print(f"Model: {model_name}")
        print(f"{'-'*60}")
        
        if not comparison['mean_relative_error']:
            print("  No common numeric columns found.")
            continue
        
        # Calculate average relative errors
        mean_errors = list(comparison['mean_relative_error'].values())
        median_errors = list(comparison['median_relative_error'].values())
        std_errors = list(comparison['std_relative_error'].values())
        
        avg_mean_error = np.mean(mean_errors) if mean_errors else 0
        avg_median_error = np.mean(median_errors) if median_errors else 0
        avg_std_error = np.mean(std_errors) if std_errors else 0
        
        print(f"  Average Mean Relative Error: {avg_mean_error:.2f}%")
        print(f"  Average Median Relative Error: {avg_median_error:.2f}%")
        print(f"  Average Std Relative Error: {avg_std_error:.2f}%")
        
        # Show top 5 columns with highest errors
        if comparison['mean_relative_error']:
            sorted_cols = sorted(
                comparison['mean_relative_error'].items(),
                key=lambda x: x[1],
                reverse=True
            )[:5]
            print(f"\n  Top 5 columns with highest mean error:")
            for col, error in sorted_cols:
                print(f"    {col}: {error:.2f}%")


STATISTICAL SIMILARITY TEST SUMMARY

Dataset: Bank

Real Data Statistics:
  Number of numeric columns: 7

------------------------------------------------------------
Model: CTGAN
------------------------------------------------------------
  Average Mean Relative Error: 3.84%
  Average Median Relative Error: 2.16%
  Average Std Relative Error: 3.25%

  Top 5 columns with highest mean error:
    balance: 14.52%
    day: 5.44%
    duration: 2.60%
    age: 2.48%
    campaign: 1.81%

------------------------------------------------------------
Model: CopulaGAN
------------------------------------------------------------
  Average Mean Relative Error: 8.29%
  Average Median Relative Error: 11.32%
  Average Std Relative Error: 11.30%

  Top 5 columns with highest mean error:
    balance: 30.37%
    day: 9.20%
    campaign: 9.01%
    duration: 5.00%
    age: 4.43%

------------------------------------------------------------
Model: GaussianCopula
---------------------------------------------

## Save Results to CSV


In [7]:
rows = []

for dataset_name, results in all_results.items():
    real_stats = results['real_stats']
    
    for model_name, comparison in results['comparisons'].items():
        for col in comparison['mean_relative_error'].keys():
            rows.append({
                'Dataset': dataset_name,
                'Model': model_name,
                'Column': col,
                'Mean_Real': real_stats['statistics'][col]['mean'],
                'Mean_Synthetic': results['synthetic_stats'][model_name]['statistics'][col]['mean'],
                'Mean_Error_%': comparison['mean_relative_error'][col],
                'Median_Real': real_stats['statistics'][col]['median'],
                'Median_Synthetic': results['synthetic_stats'][model_name]['statistics'][col]['median'],
                'Median_Error_%': comparison['median_relative_error'][col],
                'Std_Real': real_stats['statistics'][col]['std'],
                'Std_Synthetic': results['synthetic_stats'][model_name]['statistics'][col]['std'],
                'Std_Error_%': comparison['std_relative_error'][col],
                'Outliers_Real': real_stats['statistics'][col]['outliers'],
                'Outliers_Synthetic': results['synthetic_stats'][model_name]['statistics'][col]['outliers'],
                'Outliers_Diff': comparison['outlier_diff'][col]
            })

if rows:
    results_df = pd.DataFrame(rows)
    results_df.to_csv('statistical_similarity_results.csv', index=False)
    print(f"\nResults saved to statistical_similarity_results.csv")
    print(f"Total rows: {len(results_df)}")
    print(f"\nFirst few rows:")
    display(results_df.head())
else:
    print("No results to save.")



Results saved to statistical_similarity_results.csv
Total rows: 192

First few rows:


,Dataset,Model,Column,Mean_Real,Mean_Synthetic,Mean_Error_%,Median_Real,Median_Synthetic,Median_Error_%,Std_Real,Std_Synthetic,Std_Error_%,Outliers_Real,Outliers_Synthetic,Outliers_Diff
0,Bank,CTGAN,age,39.7254,40.7105,2.479774,38.0,40.0,5.263158,9.327813,9.466185,1.483430,0,0,0
1,Bank,CTGAN,campaign,2.6193,2.6668,1.813462,2.0,2.0,0.000000,2.997844,2.605006,13.104005,582,704,122
2,Bank,CTGAN,previous,0.0000,0.0000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0,0,0
3,Bank,CTGAN,balance,1117.0699,1279.3121,14.523908,355.0,335.0,5.633803,2647.588498,2720.165131,2.741235,1108,1487,379
4,Bank,CTGAN,duration,262.1082,268.9157,2.597210,189.0,181.0,4.232804,252.148595,264.889323,5.052865,731,857,126


## Detailed Statistics View

View detailed statistics for any dataset and model combination


In [8]:
# Example: View detailed comparison for a specific dataset and model
# Change these variables to view different combinations
dataset_to_view = 'Cancer'  # Options: 'Bank', 'Cancer', 'Alzhimers'
model_to_view = 'CTGAN'  # Options: 'CTGAN', 'CopulaGAN', 'GaussianCopula', 'TVAE'

if dataset_to_view in all_results and model_to_view in all_results[dataset_to_view]['comparisons']:
    comparison = all_results[dataset_to_view]['comparisons'][model_to_view]
    real_stats = all_results[dataset_to_view]['real_stats']
    synth_stats = all_results[dataset_to_view]['synthetic_stats'][model_to_view]
    
    # Create a comparison DataFrame
    comparison_data = []
    for col in comparison['mean_relative_error'].keys():
        comparison_data.append({
            'Column': col,
            'Mean_Real': real_stats['statistics'][col]['mean'],
            'Mean_Synthetic': synth_stats['statistics'][col]['mean'],
            'Mean_Error_%': comparison['mean_relative_error'][col],
            'Median_Real': real_stats['statistics'][col]['median'],
            'Median_Synthetic': synth_stats['statistics'][col]['median'],
            'Median_Error_%': comparison['median_relative_error'][col],
            'Std_Real': real_stats['statistics'][col]['std'],
            'Std_Synthetic': synth_stats['statistics'][col]['std'],
            'Std_Error_%': comparison['std_relative_error'][col],
            'Outliers_Real': real_stats['statistics'][col]['outliers'],
            'Outliers_Synthetic': synth_stats['statistics'][col]['outliers'],
            'Outliers_Diff': comparison['outlier_diff'][col]
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print(f"Detailed Statistics: {dataset_to_view} - {model_to_view}")
    print("="*80)
    display(comparison_df)
else:
    print(f"Dataset '{dataset_to_view}' or model '{model_to_view}' not found in results.")
    print(f"Available datasets: {list(all_results.keys())}")
    if dataset_to_view in all_results:
        print(f"Available models for {dataset_to_view}: {list(all_results[dataset_to_view]['comparisons'].keys())}")


Detailed Statistics: Cancer - CTGAN


,Column,Mean_Real,Mean_Synthetic,Mean_Error_%,Median_Real,Median_Synthetic,Median_Error_%,Std_Real,Std_Synthetic,Std_Error_%,Outliers_Real,Outliers_Synthetic,Outliers_Diff
0,smoothness_se,0.007041,0.007040,0.019894,0.006380,0.006942,8.808777,0.003003,0.003205,6.756456,30,7,23
1,compactness_worst,0.254265,0.309702,21.802879,0.211900,0.288260,36.035866,0.157336,0.175857,11.771127,16,11,5
2,concave points_se,0.011796,0.012944,9.730674,0.010930,0.012524,14.583715,0.006170,0.007953,28.895617,19,7,12
3,perimeter_worst,107.261213,115.443743,7.628602,97.660000,108.020000,10.608233,33.602542,40.302007,19.937376,15,33,18
4,area_se,40.337079,61.722680,53.017228,24.530000,39.155000,59.620872,45.491006,53.128879,16.789853,65,21,44
5,concavity_worst,0.272188,0.284566,4.547520,0.226700,0.235901,4.058668,0.208624,0.225090,7.892378,12,17,5
6,symmetry_mean,0.181162,0.171865,5.131882,0.179200,0.170100,5.078125,0.027414,0.034646,26.378923,15,7,8
7,fractal_dimension_mean,0.062798,0.058278,7.197530,0.061540,0.057530,6.516087,0.007060,0.007054,0.092448,15,11,4
8,compactness_se,0.025478,0.022422,11.996963,0.020450,0.017504,14.405868,0.017908,0.018348,2.453460,28,29,1
9,concavity_se,0.031894,0.045452,42.509323,0.025890,0.037284,44.007725,0.030186,0.030823,2.108884,22,41,19


## Automatic Execution

Run all cells above first, then execute this cell to run the complete pipeline automatically.


In [9]:
def run_complete_pipeline():
    """
    Main function to run the complete statistical similarity test pipeline.
    This function executes all steps automatically.
    """
    print("="*80)
    print("Statistical Similarity Test: Real vs Synthetic Data")
    print("Automatic Execution Started")
    print("="*80)
    
    # Step 1: Load datasets
    print("\n[Step 1/4] Loading datasets...")
    datasets_path = Path('Datasets')
    datasets = {}
    
    # Load Bank dataset (CSV)
    bank_path = datasets_path / 'bank-full.csv'
    if bank_path.exists():
        print(f"Loading Bank dataset from {bank_path}...")
        bank_data = pd.read_csv(bank_path, sep=';')
        if len(bank_data) > 10000:
            bank_data = bank_data.head(10000).reset_index(drop=True)
        datasets['Bank'] = bank_data
        print(f"Bank dataset loaded: {bank_data.shape}")
    else:
        print(f"Warning: Bank dataset not found at {bank_path}")
    
    # Load Cancer dataset (CSV)
    cancer_path = datasets_path / 'Cancer.csv'
    if cancer_path.exists():
        print(f"Loading Cancer dataset from {cancer_path}...")
        cancer_data = pd.read_csv(cancer_path)
        drop_cols = [c for c in ['id', 'ID', 'Unnamed: 0', 'Unnamed: 32'] if c in cancer_data.columns]
        if drop_cols:
            cancer_data = cancer_data.drop(columns=drop_cols)
        if 'diagnosis' in cancer_data.columns:
            cancer_data.replace({'diagnosis': {'M': 1, 'B': 0}}, inplace=True)
        datasets['Cancer'] = cancer_data
        print(f"Cancer dataset loaded: {cancer_data.shape}")
    else:
        print(f"Warning: Cancer dataset not found at {cancer_path}")
    
    # Load Alzhimers dataset (XLSX)
    alzhimers_path = datasets_path / 'Alzhimers.xlsx'
    if alzhimers_path.exists():
        print(f"Loading Alzhimers dataset from {alzhimers_path}...")
        alzhimers_data = pd.read_excel(alzhimers_path)
        drop_cols = [c for c in ['Subject ID', 'MRI ID', 'Hand', 'M/F', 'Group'] if c in alzhimers_data.columns]
        if drop_cols:
            alzhimers_data = alzhimers_data.drop(columns=drop_cols)
        alzhimers_data = alzhimers_data.fillna(alzhimers_data.median(numeric_only=True))
        datasets['Alzhimers'] = alzhimers_data
        print(f"Alzhimers dataset loaded: {alzhimers_data.shape}")
    else:
        print(f"Warning: Alzhimers dataset not found at {alzhimers_path}")
    
    print(f"\nTotal datasets loaded: {len(datasets)}")
    
    if not datasets:
        print("No datasets loaded. Exiting.")
        return None
    
    # Step 2: Define models
    print("\n[Step 2/4] Initializing SDV models...")
    models = {
        'CTGAN': CTGANSynthesizer,
        'CopulaGAN': CopulaGANSynthesizer,
        'GaussianCopula': GaussianCopulaSynthesizer,
        'TVAE': TVAESynthesizer
    }
    print(f"Models initialized: {list(models.keys())}")
    
    # Step 3: Process each dataset
    print("\n[Step 3/4] Processing datasets and generating synthetic data...")
    all_results = {}
    
    for dataset_name, real_data in datasets.items():
        print(f"\n{'#'*60}")
        print(f"Processing Dataset: {dataset_name}")
        print(f"{'#'*60}")
        
        # Calculate statistics for real data
        print(f"Calculating statistics for real {dataset_name} data...")
        real_stats = calculate_statistics(real_data, dataset_name, 'real')
        
        # Create metadata
        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(real_data)
        
        # Generate synthetic data for each model
        synthetic_data = {}
        num_samples = len(real_data)
        
        print(f"\nGenerating synthetic data for {dataset_name}...")
        
        for model_name, ModelClass in models.items():
            print(f"  Fitting {model_name}...")
            try:
                model = ModelClass(metadata=metadata)
                model.fit(real_data)
                
                print(f"  Sampling {num_samples} rows from {model_name}...")
                synthetic_data[model_name] = model.sample(num_rows=num_samples)
                
                print(f"  {model_name} completed successfully!")
            except Exception as e:
                print(f"  Error with {model_name}: {str(e)}")
                synthetic_data[model_name] = None
        
        # Store results for this dataset
        dataset_results = {
            'real_stats': real_stats,
            'synthetic_data': {},
            'synthetic_stats': {},
            'comparisons': {}
        }
        
        # Evaluate each synthetic dataset
        for model_name, synth_data in synthetic_data.items():
            if synth_data is None:
                continue
            
            print(f"  Evaluating {model_name}...")
            
            # Calculate statistics for synthetic data
            synth_stats = calculate_statistics(
                synth_data, dataset_name, f'synthetic_{model_name}'
            )
            
            # Compare statistics
            comparison = compare_statistics(real_stats, synth_stats)
            
            dataset_results['synthetic_data'][model_name] = synth_data
            dataset_results['synthetic_stats'][model_name] = synth_stats
            dataset_results['comparisons'][model_name] = comparison
        
        all_results[dataset_name] = dataset_results
        print(f"Completed processing {dataset_name}")
    
    # Step 4: Generate summary and save results
    print("\n[Step 4/4] Generating summary and saving results...")
    
    # Print summary
    print("\n" + "="*80)
    print("STATISTICAL SIMILARITY TEST SUMMARY")
    print("="*80)
    
    for dataset_name, results in all_results.items():
        print(f"\n{'='*80}")
        print(f"Dataset: {dataset_name}")
        print(f"{'='*80}")
        
        real_stats = results['real_stats']
        print(f"\nReal Data Statistics:")
        print(f"  Number of numeric columns: {len(real_stats['statistics'])}")
        
        for model_name, comparison in results['comparisons'].items():
            print(f"\n{'-'*60}")
            print(f"Model: {model_name}")
            print(f"{'-'*60}")
            
            if not comparison['mean_relative_error']:
                print("  No common numeric columns found.")
                continue
            
            # Calculate average relative errors
            mean_errors = list(comparison['mean_relative_error'].values())
            median_errors = list(comparison['median_relative_error'].values())
            std_errors = list(comparison['std_relative_error'].values())
            
            avg_mean_error = np.mean(mean_errors) if mean_errors else 0
            avg_median_error = np.mean(median_errors) if median_errors else 0
            avg_std_error = np.mean(std_errors) if std_errors else 0
            
            print(f"  Average Mean Relative Error: {avg_mean_error:.2f}%")
            print(f"  Average Median Relative Error: {avg_median_error:.2f}%")
            print(f"  Average Std Relative Error: {avg_std_error:.2f}%")
    
    # Save results to CSV
    rows = []
    
    for dataset_name, results in all_results.items():
        real_stats = results['real_stats']
        
        for model_name, comparison in results['comparisons'].items():
            for col in comparison['mean_relative_error'].keys():
                rows.append({
                    'Dataset': dataset_name,
                    'Model': model_name,
                    'Column': col,
                    'Mean_Real': real_stats['statistics'][col]['mean'],
                    'Mean_Synthetic': results['synthetic_stats'][model_name]['statistics'][col]['mean'],
                    'Mean_Error_%': comparison['mean_relative_error'][col],
                    'Median_Real': real_stats['statistics'][col]['median'],
                    'Median_Synthetic': results['synthetic_stats'][model_name]['statistics'][col]['median'],
                    'Median_Error_%': comparison['median_relative_error'][col],
                    'Std_Real': real_stats['statistics'][col]['std'],
                    'Std_Synthetic': results['synthetic_stats'][model_name]['statistics'][col]['std'],
                    'Std_Error_%': comparison['std_relative_error'][col],
                    'Outliers_Real': real_stats['statistics'][col]['outliers'],
                    'Outliers_Synthetic': results['synthetic_stats'][model_name]['statistics'][col]['outliers'],
                    'Outliers_Diff': comparison['outlier_diff'][col]
                })
    
    if rows:
        results_df = pd.DataFrame(rows)
        results_df.to_csv('statistical_similarity_results.csv', index=False)
        print(f"\n{'='*80}")
        print("Results saved to statistical_similarity_results.csv")
        print(f"Total rows: {len(results_df)}")
        print(f"{'='*80}")
        
        # Display first few rows
        print("\nFirst 10 rows of results:")
        display(results_df.head(10))
    else:
        print("No results to save.")
    
    print("\n" + "="*80)
    print("Evaluation Complete!")
    print("="*80)
    
    return all_results

# Execute the complete pipeline
print("Starting automatic execution...")
all_results = run_complete_pipeline()


Starting automatic execution...
Statistical Similarity Test: Real vs Synthetic Data
Automatic Execution Started

[Step 1/4] Loading datasets...
Loading Bank dataset from Datasets\bank-full.csv...
Bank dataset loaded: (10000, 17)
Loading Cancer dataset from Datasets\Cancer.csv...
Cancer dataset loaded: (569, 31)
Loading Alzhimers dataset from Datasets\Alzhimers.xlsx...
Alzhimers dataset loaded: (373, 10)

Total datasets loaded: 3

[Step 2/4] Initializing SDV models...
Models initialized: ['CTGAN', 'CopulaGAN', 'GaussianCopula', 'TVAE']

[Step 3/4] Processing datasets and generating synthetic data...

############################################################
Processing Dataset: Bank
############################################################
Calculating statistics for real Bank data...

Generating synthetic data for Bank...
  Fitting CTGAN...
  Sampling 10000 rows from CTGAN...
  CTGAN completed successfully!
  Fitting CopulaGAN...
  Sampling 10000 rows from CopulaGAN...
  CopulaGAN 

,Dataset,Model,Column,Mean_Real,Mean_Synthetic,Mean_Error_%,Median_Real,Median_Synthetic,Median_Error_%,Std_Real,Std_Synthetic,Std_Error_%,Outliers_Real,Outliers_Synthetic,Outliers_Diff
0,Bank,CTGAN,age,39.7254,41.1644,3.622368,38.0,40.0,5.263158,9.327813,9.184814,1.533039,0,0,0
1,Bank,CTGAN,campaign,2.6193,2.3537,10.140114,2.0,2.0,0.000000,2.997844,2.059277,31.308069,582,280,302
2,Bank,CTGAN,previous,0.0000,0.0000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0,0,0
3,Bank,CTGAN,balance,1117.0699,1395.0201,24.882078,355.0,346.0,2.535211,2647.588498,3139.675017,18.586216,1108,1058,50
4,Bank,CTGAN,duration,262.1082,305.7708,16.658235,189.0,208.0,10.052910,252.148595,305.213532,21.045105,731,843,112
5,Bank,CTGAN,day,15.0585,14.4226,4.222864,14.0,13.0,7.142857,9.038598,9.514098,5.260776,0,0,0
6,Bank,CTGAN,pdays,-1.0000,-1.0000,0.000000,-1.0,-1.0,0.000000,0.000000,0.000000,0.000000,0,0,0
7,Bank,CopulaGAN,age,39.7254,41.3188,4.011036,38.0,41.0,7.894737,9.327813,9.402830,0.804227,0,0,0
8,Bank,CopulaGAN,campaign,2.6193,2.1308,18.650021,2.0,2.0,0.000000,2.997844,2.235485,25.430248,582,1052,470
9,Bank,CopulaGAN,previous,0.0000,0.0000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0,0,0



Evaluation Complete!
